# 🧬 HAPLO-BENCH : Générateur d'instances de benchmark

Ce notebook interactif vous guide pas à pas dans la génération de jeux de données synthétiques pour l'inférence d'haplotypes (détection de communautés sous modèle bayésien signé).

**Étapes clés :**
1. **Installation** de la bibliothèque et clonage du projet.
2. **Configuration** des paramètres biologiques (chromosome, densité de variants) et technologiques (couverture, type de séquençage).
3. **Génération** en boucle d'instances indépendantes avec des graines aléatoires (*seeds*) consécutives.
4. **Visualisation & Vérification** des fichiers biologiques et statistiques générés.

---

## 🛠️ Étape 1 : Installation de HAPLO-BENCH

Exécutez cette cellule pour cloner le dépôt et installer la bibliothèque `haplo-bench` en mode éditable.

In [ ]:
#@title 📥 Installation des packages
import os
import sys

print("📥 Clonage du dépôt Haplotypes depuis GitHub...")
if not os.path.exists("Haplotypes"):
    !git clone https://github.com/Ludwig-H/Haplotypes.git
    %cd Haplotypes
else:
    print("Le dossier Haplotypes existe déjà.")
    %cd Haplotypes

print("⚙️ Installation de la bibliothèque haplo-bench en mode éditable...")
!pip install -e .

print("✅ Installation terminée avec succès !")

## ⚙️ Étape 2 : Configuration de la Simulation

Définissez les paramètres de votre benchmark à l'aide du formulaire interactif ci-dessous.

### Explications des paramètres :
*   `preset` : Profil de séquençage à émuler (ex: PacBio HiFi, Illumina court-reads, Oxford Nanopore Q20, duplex ou ultralong).
*   `chromosome` : Chromosome humain cible. Définit la longueur physique $L$ (de `chr22` le plus court à `chr1` le plus long).
*   `density` ($ho_{\text{het}}$) : Densité de variants hétérozygotes (SNPs) le long de la référence. Plus elle est faible, plus le graphe sera clairsémé.
*   `coverage` : Profondeur ou couverture cible du séquençage (par défaut 30x).
*   `min_shared_variants` ($m_{\min}$) : Nombre minimum de variants requis pour créer une arête entre deux lectures.
*   `num_graphs` : Nombre d'instances à générer (avec des graines aléatoires consécutives de $1$ à $N$).
*   `output_dir` : Répertoire de destination pour stocker les fichiers générés.

In [ ]:
#@title 🎛️ Formulaire de Configuration { display-mode: "form" }

preset = "pacbio_hifi" #@param ["theory_fixed", "illumina_pe150", "pacbio_hifi", "ont_q20", "ont_duplex", "ont_ultralong", "hybrid_illumina_pacbio", "hybrid_illumina_ont"]
chromosome = "chr22" #@param ["chr20", "chr22", "chr1", "chr2"]
density = 0.00075 #@param {type:"number"}
coverage = 30 #@param {type:"integer"}
min_shared_variants = 1 #@param {type:"integer"}
num_graphs = 5 #@param {type:"integer"}
output_dir = "data/benchmark" #@param {type:"string"}

print("✨ Paramètres enregistrés avec succès !")
print(f"  - Technologie : {preset}")
print(f"  - Cible : {chromosome} (variants : {density})")
print(f"  - Couverture : {coverage}x")
print(f"  - Arêtes (seuil) : {min_shared_variants} variant(s) partagé(s)")
print(f"  - Quantité : {num_graphs} instances")

## 🚀 Étape 3 : Génération en boucle des Graphes

Exécutez cette cellule pour générer les instances. Pour chaque instance :
1. Une graine aléatoire distincte (*seed*) allant de $1$ à `num_graphs` est injectée.
2. Un fichier YAML de configuration temporaire est écrit.
3. L'outil `haplo-bench generate` est invoqué en arrière-plan.

In [ ]:
#@title ⚙️ Lancement de la génération
import yaml
import os
import subprocess

# Création du dossier parent si nécessaire
os.makedirs(output_dir, exist_ok=True)

# Dictionnaire de base de la configuration
base_config = {
    "preset": preset,
    "reference": {
        "assembly": "GRCh38.p14",
        "chromosome": chromosome
    },
    "variants": {
        "model": "bernoulli",
        "density": density
    },
    "coverage": {
        "target": coverage,
        "compute_n_reads": True
    },
    "graph": {
        "min_shared_variants": min_shared_variants,
        "edge_rule": "likelihood_ratio"
    }
}

print(f"🚀 Début de la génération de {num_graphs} instances...")

for seed in range(1, num_graphs + 1):
    print(f"\n" + "="*60)
    print(f"▶️ Instance {seed} / {num_graphs} (Graine : {seed})")
    print("="*60)
    
    # Injecter la graine actuelle
    config = base_config.copy()
    config["seed"] = seed
    
    # Écriture du fichier temporaire
    temp_yaml = f"temp_config_seed_{seed}.yaml"
    with open(temp_yaml, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False)
        
    instance_path = os.path.join(output_dir, f"instance_seed_{seed}")
    
    # Construction de la commande
    cmd = [
        "haplo-bench", "generate",
        "--config", temp_yaml,
        "--out", instance_path,
        "--seed", str(seed)
    ]
    
    print(f"Exécution : {' '.join(cmd)}")
    
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(res.stdout)
        print(f"✅ Instance générée dans : {instance_path}")
    except subprocess.CalledProcessError as e:
        # Fallback si l'outil n'accepte la graine que dans la configuration YAML
        print(f"⚠️ Échec direct du CLI : {e.stderr.strip()}")
        print("🔄 Tentative de repli via la seed intégrée au YAML...")
        cmd_fallback = [
            "haplo-bench", "generate",
            "--config", temp_yaml,
            "--out", instance_path
        ]
        try:
            res_fb = subprocess.run(cmd_fallback, capture_output=True, text=True, check=True)
            print(res_fb.stdout)
            print(f"✅ Instance générée (via YAML) dans : {instance_path}")
        except subprocess.CalledProcessError as e_fb:
            print(f"❌ Échec définitif : {e_fb.stderr.strip()}")
            
    # Nettoyage du fichier temporaire
    if os.path.exists(temp_yaml):
        os.remove(temp_yaml)

print("\n🎉 Génération de toutes les instances terminée !")

## 📊 Étape 4 : Visualisation & Vérification des Sorties

Exécutez cette cellule pour inspecter les résultats de la première instance (Seed 1) :
1. **Rapport statistique** : Affiche les métriques clés sous forme de tableau.
2. **Aperçu du Graphe** : Affiche les premières lignes du fichier `graph/edges.tsv`.
3. **Arborescence des fichiers** : Liste tous les fichiers biologiques, graphiques et de vérité terrain générés pour valider le bon déroulement de l'opération.

In [ ]:
#@title 🔍 Inspecter les fichiers et métriques
import pandas as pd
import json
import glob
import os
from IPython.display import HTML, display

instances = sorted(glob.glob(os.path.join(output_dir, "instance_seed_*")))

if instances:
    first_inst = instances[0]
    print(f"🔎 Analyse de l'instance de test : {first_inst}\n")
    
    # 1. Lire le rapport sommaire JSON et l'afficher en tableau Markdown/HTML
    summary_json = os.path.join(first_inst, "report", "summary.json")
    if os.path.exists(summary_json):
        with open(summary_json, "r", encoding="utf-8") as f:
            metrics = json.load(f)
        
        print("📈 Métriques Clés de l'Instance :")
        # Formatage en tableau HTML simple pour Colab
        html_table = "<table style='width:50%; border-collapse: collapse; border: 1px solid #ddd;'>"
        html_table += "<tr style='background-color: #f2f2f2;'><th style='padding: 8px; border: 1px solid #ddd; text-align: left;'>Métrique</th><th style='padding: 8px; border: 1px solid #ddd; text-align: left;'>Valeur</th></tr>"
        for k, v in metrics.items():
            html_table += f"<tr><td style='padding: 8px; border: 1px solid #ddd;'><b>{k}</b></td><td style='padding: 8px; border: 1px solid #ddd;'>{v}</td></tr>"
        html_table += "</table>"
        display(HTML(html_table))
        print()
        
    # 2. Charger les arêtes signées et pondérées
    edges_tsv = os.path.join(first_inst, "graph", "edges.tsv")
    if os.path.exists(edges_tsv):
        df = pd.read_csv(edges_tsv, sep="\t")
        print(f"📊 Aperçu des premières arêtes du graphe signé pondéré ({len(df)} arêtes totales) :")
        # Expliquer brièvement les colonnes
        print("  (Note: weight > 0 = attraction [mêmes communautés], weight < 0 = répulsion)")
        display(df.head())
        print()
        
    # 3. Vérifier et lister l'arborescence des fichiers générés
    print("📁 Fichiers générés par le simulateur :")
    for group in ["bio", "truth", "graph", "report"]:
        group_dir = os.path.join(first_inst, group)
        if os.path.exists(group_dir):
            files = sorted(os.listdir(group_dir))
            print(f"  📂 {group}/ :")
            for file in files:
                size = os.path.getsize(os.path.join(group_dir, file))
                print(f"    📄 {file:<25} ({size} octets)")
else:
    print("❌ Aucune instance générée trouvée. Veuillez lancer l'étape 3 d'abord.")